In [29]:
try:
    !python -m pip install -q -r requirements.txt
    print("Installation of dependencies completed successfully.")
except Exception as e:
    print(f"An error occurred while installing dependencies: {e}")

import re 
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import pipeline
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

Installation of dependencies completed successfully.


In [30]:
df = pd.read_csv('Dataset_MBG.csv')
df.shape

(29965, 8)

## **Preprocessing Data**
5 Steps of Text Preprocessing : 
1 - Cleaning : Clean texts from URL, hastaghs, character, etc.
2 - Case Folding : Lower all alphabet character
3 - Tokenization : Change it into tokens
4 - Stopword Removal : Remove stopwords / un-important words with no meaning
5 - Stemming : Change slang words into formal 
6 - Token join : Join all the tokens (words)

In [31]:
def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # hapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # hapus hashtag
    text = re.sub(r'RT[\s]', '', text) # hapus RT
    text = re.sub(r"http\S+", '', text) # hapus link
    text = re.sub(r'[0-9]+', '', text) # hapus angka
    text = re.sub(r'[^\w\s]', '', text) # hapus karakter selain huruf dan angka
    text = text.lower() # mengkonversi ke huruf kecil
    text = re.sub(r'\s+', ' ', text) # hapus spasi ganda/multiple
    text = text.strip() # hapus spasi di awal dan akhir
    return text

df['text_cleaning'] = df['text'].apply(cleaningText)
df[['text','text_cleaning']].head(3)

,text,text_cleaning
0,@JukiHoki Problem di Indonesia itu bukan kelap...,problem di indonesia itu bukan kelaparan tapi ...
1,@democrazymedia Sedang mikir keras..gimana IKN...,sedang mikir kerasgimana ikn bisa berlanjut se...
2,Makan bergizi gratis di Kota Tangerang bulan O...,makan bergizi gratis di kota tangerang bulan o...


In [32]:
# Load Indonesian stopwords
with open('Stopwords-indonesia.txt', 'r') as file:
    stopwords = set(file.read().split())

# Tokenize and remove stopwords
def remove_stopwords(tokens):
    tokens = [word for word in tokens if word not in stopwords]  # remove stopwords
    return ' '.join(tokens)

# Tokenization
df['text_tokens'] = df['text_cleaning'].apply(lambda x: x.split())
# Remove stopwords
df['no_stopwords_text'] = df['text_tokens'].apply(remove_stopwords)
df[['text_cleaning', 'text_tokens', 'no_stopwords_text']].head(3)

,text_cleaning,text_tokens,no_stopwords_text
0,problem di indonesia itu bukan kelaparan tapi ...,"[problem, di, indonesia, itu, bukan, kelaparan...",problem indonesia kelaparan pendidikan berdasa...
1,sedang mikir kerasgimana ikn bisa berlanjut se...,"[sedang, mikir, kerasgimana, ikn, bisa, berlan...",mikir kerasgimana ikn berlanjut jualan preside...
2,makan bergizi gratis di kota tangerang bulan o...,"[makan, bergizi, gratis, di, kota, tangerang, ...",makan bergizi gratis kota tangerang oktober sa...


In [33]:
# Normalize informal/slang words using kamus-alay
kamus_df = pd.read_csv('Colloquial-indonesian-lexicon.csv')
kamus = dict(zip(kamus_df['slang'].astype(str).str.lower(), kamus_df['formal'].astype(str)))

def normalize_alay(text):
    """Normalize informal Indonesian words to formal ones"""
    words = text.split()
    normalized = []
    for word in words:
        # Check if word is in informal dictionary, replace if found
        normalized.append(kamus.get(word, word))
    return ' '.join(normalized)

# Apply normalization
df['normalized_alay'] = df['no_stopwords_text'].apply(normalize_alay)
df[['no_stopwords_text', 'normalized_alay']].head(3)

,no_stopwords_text,normalized_alay
0,problem indonesia kelaparan pendidikan berdasa...,problem indonesia kelaparan pendidikan berdasa...
1,mikir kerasgimana ikn berlanjut jualan preside...,mikir kerasgimana ikn berlanjut jualan preside...
2,makan bergizi gratis kota tangerang oktober sa...,makan bergizi gratis kota tangerang oktober sa...


In [34]:
# Stemming using sastrawi
# 1 - Initialize stemmer 
stemmer = StemmerFactory().create_stemmer()

# 2 - Begin stemming process
df['stemmed_text'] = df['normalized_alay'].apply(stemmer.stem)
df['stemmed_text'].head(3)

0    problem indonesia lapar didik dasar penting ha...
1    mikir kerasgimana ikn lanjut jual presiden yan...
2    makan gizi gratis kota tangerang oktober sasar...
Name: stemmed_text, dtype: str

In [56]:
# Load model pipeline 
pipeline_sentiment = pipeline(
    "text-classification",
    model="mdhugol/indonesia-bert-sentiment-classification",
)

# sentiment label index 
label_index = {'LABEL_0': 'positive', 'LABEL_1': 'neutral', 'LABEL_2': 'negative'}

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 534.56it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: mdhugol/indonesia-bert-sentiment-classification
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [41]:
# Take sample data for labelling 
train = df.iloc[:1000].copy()

# Apply sentiment analysis to the training set
text_clf = train['stemmed_text'].apply(pipeline_sentiment)

result = [label_index[label[0]['label']] for label in text_clf] 
score = [label[0]['score'] for label in text_clf] 

# Assign result 
train['sentiment_label'], train['sentiment_score'] = result, score

In [54]:
def train_sample(idx):
    print(f"Text - {train['text'][idx]}")
    print(f"Text Label: {train['sentiment_label'][idx]}")
    print(f"Label Score: {train['sentiment_score'][idx]}")

train_sample(6)

Text - mereka juga dapat makan siang gratis sih https://t.co/OL63mFUPWx
Text Label: positive
Label Score: 0.9495764970779419


In [50]:
train['sentiment_label'].value_counts()

sentiment_label
neutral     506
positive    250
negative    244
Name: count, dtype: int64

In [61]:
# Check Positive Sentiment Sample
display(train[train['sentiment_label'] == 'positive'].sample(5).loc[
    :,['text','sentiment_label','sentiment_score']
])

,text,sentiment_label,sentiment_score
250,Pelaksanaan Program Makan Siang Gratis Berkola...,positive,0.844982
217,Namiss ko SMG at MBG 🥹,positive,0.597364
243,Pelaksanaan Program Makan Siang Gratis Berkola...,positive,0.844982
248,Pelaksanaan Program Makan Siang Gratis Berkola...,positive,0.844982
694,@chaeyouzzg @pagersoobin @1 makan siang gratis...,positive,0.969194
